# Le tableau de trace
*NSI — Suivre un programme à la main (10 minutes)*

Au contrôle, pas d'ordinateur : pour savoir ce que fait un programme, on l'exécute **à la main** avec un tableau de trace.

**Le principe :**
- **une colonne par variable** (plus, si besoin, une colonne pour compter quelque chose) ;
- **une ligne = une photo** de toutes les variables à un instant donné ;
- on avance **ligne par ligne**, et on **recopie** les valeurs qui ne changent pas.

---
## 1. Une boucle

Une première ligne **avant** la boucle, puis **une ligne par tour** (valeurs à la fin du tour).

`range(n)` donne `0, 1, …, n - 1` : la boucle fait `n` tours.

In [1]:
def power_bf(x, n):
    p = 1
    for i in range(n):
        p = p * x
    return p

`power_bf(2, 4)`, avec une colonne qui compte les multiplications :

| Étape | `i` | `p` | multiplications |
|:---|:---:|:---:|:---:|
| avant la boucle | — | 1 | 0 |
| tour 1 | 0 | 2 | 1 |
| tour 2 | 1 | 4 | 2 |
| tour 3 | 2 | 8 | 3 |
| tour 4 | 3 | 16 | 4 |

Une multiplication par tour, et `n` tours : **`n` multiplications**. On fait un petit exemple, puis on **généralise**.

---
## 2. Une boucle `while` avec des `if` : la dichotomie

Même principe, avec une colonne pour la **décision** prise par le `if`. `t[3]` est la case d'indice 3 (on compte à partir de 0) ; `//` est la division entière.

In [2]:
def recherche(t, x):
    gauche = 0
    droite = len(t) - 1
    while gauche <= droite:
        milieu = (gauche + droite) // 2
        if t[milieu] == x:
            return True
        elif t[milieu] < x:
            gauche = milieu + 1
        else:
            droite = milieu - 1
    return False

`recherche([2, 5, 7, 9, 12, 15, 20], 15)` :

| Étape | `gauche` | `droite` | `milieu` | `t[milieu]` | décision |
|:---|:---:|:---:|:---:|:---:|:---|
| avant la boucle | 0 | 6 | — | — | on entre |
| tour 1 | 4 | 6 | 3 | 9 | 9 < 15 : à droite, `gauche = 4` |
| tour 2 | 4 | 6 | 5 | 15 | trouvé : **renvoie `True`** |

Si la valeur n'est pas dans le tableau, on s'arrête quand `gauche` dépasse `droite` (zone vide) : la fonction renvoie `False`.

---
## 3. Une fonction récursive

Une fonction récursive **s'appelle elle-même**. Chaque appel **attend** la réponse du suivant.

**Le tableau :** une **ligne par appel**. On remplit en **descendant** ⬇️ jusqu'au cas de base, puis la colonne « renvoie » en **remontant** ⬆️.

`n % 2 == 0` veut dire « `n` est pair ».

In [3]:
def power_dac(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        y = power_dac(x, n // 2)
        return y * y
    else:
        return x * power_dac(x, n - 1)

`power_dac(2, 10)` :

| Appel | `n` | ce qu'il faut calculer | multiplications | renvoie (en remontant ⬆️) |
|:---|:---:|:---|:---:|:---:|
| `power_dac(2, 10)` ⬇️ | 10 (pair) | `y = power_dac(2, 5)`, puis `y * y` | 1 | 32 × 32 = **1024** |
| `power_dac(2, 5)` ⬇️ | 5 (impair) | `2 * power_dac(2, 4)` | 1 | 2 × 16 = 32 |
| `power_dac(2, 4)` ⬇️ | 4 (pair) | `y = power_dac(2, 2)`, puis `y * y` | 1 | 4 × 4 = 16 |
| `power_dac(2, 2)` ⬇️ | 2 (pair) | `y = power_dac(2, 1)`, puis `y * y` | 1 | 2 × 2 = 4 |
| `power_dac(2, 1)` | 1 | cas de base | 0 | 2 |

Résultat : **1024**, avec **4 multiplications** (la somme de la colonne), contre 10 pour `power_bf`.

⚠️ On ne peut pas calculer « renvoie » en descendant : on ne connaît la réponse qu'en partant du bas.

In [4]:
# Vérification par l'ordinateur
def power_dac_trace(x, n, decalage):
    print(decalage + "appel power_dac(" + str(x) + ", " + str(n) + ")")
    if n == 1:
        print(decalage + "cas de base : renvoie " + str(x))
        return x
    if n % 2 == 0:
        y = power_dac_trace(x, n // 2, decalage + "    ")
        resultat = y * y
    else:
        resultat = x * power_dac_trace(x, n - 1, decalage + "    ")
    print(decalage + "power_dac(" + str(x) + ", " + str(n) + ") renvoie " + str(resultat))
    return resultat


power_dac_trace(2, 10, "")

appel power_dac(2, 10)
    appel power_dac(2, 5)
        appel power_dac(2, 4)
            appel power_dac(2, 2)
                appel power_dac(2, 1)
                cas de base : renvoie 2
            power_dac(2, 2) renvoie 4
        power_dac(2, 4) renvoie 16
    power_dac(2, 5) renvoie 32
power_dac(2, 10) renvoie 1024


1024

---
## 4. Deux appels : l'arbre des appels

Si un appel en lance **deux**, on dessine un **arbre** : sous chaque appel, les appels qu'il fait. Voici la version maladroite de `power_dac` :
```python
return power_dac(x, n // 2) * power_dac(x, n // 2)
```
Pour `n = 4` :
```
               power_dac(x, 4)               1 multiplication
              /               \
      power_dac(x, 2)    power_dac(x, 2)     1 + 1
        /       \          /       \
    (x, 1)    (x, 1)    (x, 1)    (x, 1)     cas de base : 0
```
On voit **deux fois le même appel avec les mêmes arguments** : on refait le même calcul pour rien.

**Le réflexe :** même appel, mêmes arguments → on le fait **une seule fois** et on **range le résultat dans une variable** (c'est le `y` de la partie 3).

---
## En résumé

| Ce qu'on suit | Le tableau |
|:---|:---|
| une boucle | une ligne avant la boucle, puis une ligne par tour |
| des `if` dans la boucle | une colonne « décision » |
| une fonction récursive (un appel) | une ligne par appel : on descend, puis on remonte |
| deux appels | un arbre des appels |

Pour compter des opérations : une **colonne qui compte**, un **petit exemple**, puis on **généralise**.